# Data exploration and audit

Audit of the Geneva building-permit export before any cleaning, splitting or modeling decision is made.

In [1]:
import sys

sys.path.insert(0, "..")

## Basic inspection
Load the export and see what is in it - shape, columns, types, first rows.

In [2]:
# Loading the data in the Pandas DataFrame - outputing the shape of the data.
from src.data import load_source

df = load_source()
df.shape

(197224, 16)

In [3]:
df.head()

,OBJECTID,ID_DOSSIER,TYPE_DOSSIER,TYPE_OPERATION,NOM_DOSSIER,NO_DOSSIER,NO_COMPLEMENTAIRE,STATUT,DATE_DEPOT,DESCRIPTION,OPERATION,ORIGINE,DATE_MAJ_2,LIEN_SAD,E,N
0,45,DD 17497/1,DD,--,Demande définitive,17497,1,-,NaT,non renseigné,Non renseigné (valeur par défaut à l'importati...,ANC,2002-02-26,<NA>,2498512.96,1125146.27
1,46,DD 87127/1,DD,--,Demande définitive,87127,1,-,NaT,non renseigné,Non renseigné (valeur par défaut à l'importati...,ANC,2002-02-26,<NA>,2498522.16,1125164.87
2,47,DD 18791/1,DD,--,Demande définitive,18791,1,-,NaT,non renseigné,Non renseigné (valeur par défaut à l'importati...,ANC,2001-06-27,<NA>,2502738.54,1117004.57
3,48,DD 14989/1,DD,--,Demande définitive,14989,1,-,NaT,non renseigné,Non renseigné (valeur par défaut à l'importati...,ANC,2001-06-27,<NA>,2502738.33,1116951.27
4,49,DD 25882/1,DD,--,Demande définitive,25882,1,-,NaT,non renseigné,Non renseigné (valeur par défaut à l'importati...,ANC,2001-06-27,<NA>,2502739.63,1116938.97


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 197224 entries, 0 to 197223
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   OBJECTID           197224 non-null  string        
 1   ID_DOSSIER         197224 non-null  string        
 2   TYPE_DOSSIER       197224 non-null  string        
 3   TYPE_OPERATION     197189 non-null  string        
 4   NOM_DOSSIER        197215 non-null  string        
 5   NO_DOSSIER         197224 non-null  string        
 6   NO_COMPLEMENTAIRE  197223 non-null  string        
 7   STATUT             197217 non-null  string        
 8   DATE_DEPOT         128809 non-null  datetime64[us]
 9   DESCRIPTION        197217 non-null  string        
 10  OPERATION          197221 non-null  string        
 11  ORIGINE            197221 non-null  string        
 12  DATE_MAJ_2         197188 non-null  datetime64[us]
 13  LIEN_SAD           128813 non-null  string        
 14 

## Missing data
'head()' shows placeholder values where data is absent. Checking how common they are, sence 'info()' only counts true nulls.

In [5]:
for col in df.select_dtypes("string").columns:
    print(col, df[col].value_counts().head(3).to_dict())

OBJECTID {'45': 1, '46': 1, '47': 1}
ID_DOSSIER {'DD 1568': 3, 'DD 104289/1': 3, 'DD 93616/1': 2}
TYPE_DOSSIER {'DD': 103662, 'APA': 79376, 'APAT': 6037}
TYPE_OPERATION {'--': 147397, 'REN': 17361, 'PI': 14815}
NOM_DOSSIER {'Demande définitive': 103660, 'Autorisation procédure accélérée': 79367, 'Autorisation par annonce de travaux': 6037}
NO_DOSSIER {'93968': 12, '92130': 12, '23014': 9}
NO_COMPLEMENTAIRE {'1': 184202, '2': 9417, '3': 2516}
STATUT {'-': 68411, 'TERMINE': 62923, 'ARCHIVE': 16943}
DESCRIPTION {'non renseigné': 68411, "rénovation d'un appartement au 2ème étage": 2148, "rénovation d'un appartement au 1er étage": 1979}
OPERATION {"Non renseigné (valeur par défaut à l'importation des données)": 147429, "Travaux d'entretien et de rénovation intérieure ou extérieure, sans création de surfaces supplémentaires destinées à l'habitat ou aux activités et sans changement d'affectation important": 17361, "Construction neuve de peu d'importance, isolée ou attenante au bâtiment princi

The top values only reveal frequent placeholders. The next three checks look for rarer ones:
every value of the low-cardinality columns, blank strings in all text columns, and all short descriptions.

In [6]:
small = [
    "TYPE_DOSSIER",
    "TYPE_OPERATION",
    "NOM_DOSSIER",
    "NO_COMPLEMENTAIRE",
    "STATUT",
    "OPERATION",
    "ORIGINE",
]

for col in small:
    print(col, df[col].nunique(), sorted(df[col].dropna().unique()))

TYPE_DOSSIER 11 ['APA', 'APAT', 'APL', 'DD', 'DP', 'DR', 'LER', 'LER-PA', 'LER-PAT', 'M', 'MPA']
TYPE_OPERATION 10 ['--', 'AFF', 'AGR', 'AGV', 'AM', 'EQU', 'NOU', 'PI', 'REN', 'VIL']
NOM_DOSSIER 12 ['Autorisation par annonce de travaux', 'Autorisation par procédure accélérée', 'Autorisation procédure accélérée', 'Demande de renseignements', 'Demande définitive', 'Demande préalable', 'Démolition', 'Démolition procédure accéléré', 'Procédure accélérée', 'Requête en approbation sur la loi des eaux et des routes', 'Requête en approbation sur la loi des eaux et des routes par annonce de travaux', 'Requête en approbation sur la loi des eaux et des routes par procédure accélérée']
NO_COMPLEMENTAIRE 18 ['0', '1', '10', '11', '12', '13', '14', '15', '16', '2', '3', '31', '4', '5', '6', '7', '8', '9']
STATUT 21 ['-', 'ABANDONNE', 'ACCEPTE', 'ARCHANT', 'ARCHIVE', 'AUTORISATION ANNULEE', 'CADUQUE', 'CHANTIER', 'CLOTURE ADMIN', 'DECIDE', 'EN SUSPENS', 'ENREGISTRE', 'ENREGISTREMENT', 'INSTRUCTION', 

In [7]:
df.select_dtypes("string").apply(lambda s: s.str.strip().eq("").sum())

OBJECTID             0
ID_DOSSIER           0
TYPE_DOSSIER         0
TYPE_OPERATION       0
NOM_DOSSIER          0
NO_DOSSIER           0
NO_COMPLEMENTAIRE    0
STATUT               0
DESCRIPTION          0
OPERATION            0
ORIGINE              0
LIEN_SAD             0
E                    0
N                    0
dtype: int64

In [8]:
d = df["DESCRIPTION"].dropna()
d[d.str.len() <= 15].value_counts()
print(d[d.str.len() <= 15].value_counts().to_string())

DESCRIPTION
non renseigné      68411
piscine              803
véranda              727
jardin d'hiver       315
jour en toiture      137
garage               133
Piscine              115
villa - garage       113
Véranda               97
villa                 79
couvert               68
abri de jardin        64
clôture               41
villa et garage       36
villa - parking       30
mur anti-bruit        29
cabane à outils       26
véranda -             24
hangar agricole       23
sas d'entrée          22
pergola               22
piscine -             20
mur                   17
portail               16
mur anti bruit        16
Garage                14
mur de clôture        13
Véranda -             11
garage -              10
villa -                9
serre de jardin        9
villa - couvert        9
abri à vélos           9
marquise               8
dragage du port        8
couvert à vélos        8
mur antibruit          8
couvert à bois         7
couvert -              6
serre        


### Placeholders found

The export uses exactly four placeholder values: `--` in `TYPE_OPERATION`, `-` in `STATUT`,
`non renseigné` in `DESCRIPTION` and `Non renseigné (valeur par défaut à l'importation des données)`
in `OPERATION`.

### Nulls vs placeholders

Counting both per column with the complete list.

In [9]:
import pandas as pd

placeholders = [
    "--",
    "-",
    "non renseigné",
    "Non renseigné (valeur par défaut à l'importation des données)",
]

pd.DataFrame({"nulls": df.isna().sum(), "placeholders": df.isin(placeholders).sum()})

,nulls,placeholders
OBJECTID,0,0
ID_DOSSIER,0,0
TYPE_DOSSIER,0,0
TYPE_OPERATION,35,147397
NOM_DOSSIER,9,0
NO_DOSSIER,0,0
NO_COMPLEMENTAIRE,1,0
STATUT,7,68411
DATE_DEPOT,68415,0
DESCRIPTION,7,68411


### Are the missing fields the same records?

Checking whether the records without a description are also the ones without a date, status and link.

In [10]:
no_text = df["DESCRIPTION"].isna() | df["DESCRIPTION"].isin(placeholders)

pd.DataFrame(
    {
        "no date": df.loc[no_text, "DATE_DEPOT"].isna().sum(),
        "no status": df.loc[no_text, "STATUT"].isin(placeholders).sum(),
        "no link": df.loc[no_text, "LIEN_SAD"].isna().sum(),
        "origin ANC": df.loc[no_text, "ORIGINE"].eq("ANC").sum(),
    },
    index=[f"records without text: {no_text.sum()}"],
)

,no date,no status,no link,origin ANC
records without text: 68418,68412,68411,68411,68408


## Labels

Setting the empty records aside and counting the operation codes among the records that do have a description.

In [11]:
with_text = df[~no_text]

with_text["TYPE_OPERATION"].value_counts(dropna=False)

TYPE_OPERATION
--     79022
REN    17361
PI     14815
EQU     4549
VIL     4139
NOU     4011
AGR     2324
AGV     1992
AFF      555
AM        38
Name: count, dtype: int64[pyarrow]

## Labels over time

Counting labelled and unlabelled records per filing year.

In [12]:
year = with_text["DATE_DEPOT"].dt.year
labelled = with_text["TYPE_OPERATION"].ne("--")

pd.crosstab(year, labelled)

TYPE_OPERATION,False,True
DATE_DEPOT,,
1995.0,1,0
1996.0,3224,11
1997.0,2716,8
1998.0,2884,19
1999.0,952,2099
2000.0,665,2380
2001.0,657,2431
2002.0,561,2469
2003.0,533,2624


## Class mix over time

Share of each operation code among labelled records, using the four calendar periods
from the research plan: up to 2010, 2011–2012, 2013 and 2014–2026.

In [13]:
labelled_rows = with_text[labelled]
period = pd.cut(
    labelled_rows["DATE_DEPOT"].dt.year,
    bins=[0, 2010, 2012, 2013, 2026],
    labels=["≤2010", "2011–2012", "2013", "2014–2026"],
)

pd.crosstab(labelled_rows["TYPE_OPERATION"], period, normalize="columns").round(3)

DATE_DEPOT,≤2010,2011–2012,2013,2014–2026
TYPE_OPERATION,,,,
AFF,0.014,0.008,0.008,0.005
AGR,0.043,0.038,0.038,0.063
AGV,0.041,0.032,0.030,0.044
AM,0.001,0.002,0.000,0.000
EQU,0.091,0.100,0.080,0.092
NOU,0.058,0.056,0.061,0.164
PI,0.320,0.302,0.389,0.209
REN,0.379,0.408,0.334,0.235
VIL,0.053,0.055,0.060,0.189


## Dossier types, origins and statuses

I count the records for each dossier code and description to check for naming variants and missing descriptions.

I then check how common each origin and status value is. Any codes with unclear meanings will need clarification before I use them to interpret the data.

In [14]:
dossier_types = (
    df.groupby(["TYPE_DOSSIER", "NOM_DOSSIER"], dropna=False)
    .size()
    .reset_index(name="records")
    .sort_values(["TYPE_DOSSIER", "records"], ascending=[True, False])
)

with pd.option_context("display.max_colwidth", None):
    display(dossier_types)

,TYPE_DOSSIER,NOM_DOSSIER,records
1,APA,Autorisation procédure accélérée,79367
0,APA,Autorisation par procédure accélérée,5
3,APA,<NA>,3
2,APA,Procédure accélérée,1
4,APAT,Autorisation par annonce de travaux,6037
5,APL,<NA>,4
6,DD,Demande définitive,103660
7,DD,<NA>,2
8,DP,Demande préalable,1552
9,DR,Demande de renseignements,18


In [15]:
for column in ["ORIGINE", "STATUT"]:
    counts = df[column].value_counts(dropna=False).to_frame(name="records")
    counts["percent"] = (100 * counts["records"] / len(df)).round(2)

    print(column)
    display(counts)

ORIGINE


,records,percent
ORIGINE,,
ANC,68408,34.69
SAD,54520,27.64
SIT,53362,27.06
AC-DEM,20931,10.61
<NA>,3,0.0


STATUT


,records,percent
STATUT,,
-,68411,34.69
TERMINE,62923,31.9
ARCHIVE,16943,8.59
ACCEPTE,12067,6.12
CLOTURE ADMIN,8933,4.53
RENVOYE,7954,4.03
CADUQUE,5591,2.83
ABANDONNE,4706,2.39
CHANTIER,3570,1.81


DD and APA account for about 92.8% of the records. The remaining dossier types include announcements of works, preliminary requests, requests for information, demolition, and requests concerning water and roads.

APA has three wording variants in `NOM_DOSSIER`. Nine records have no description: four APL, three APA and two DD. This table does not establish what APL means.

`ORIGINE` contains four codes—ANC, SAD, SIT and AC-DEM—and three missing values. I still need to confirm what these codes represent.

The status placeholder `-` accounts for 34.69% of records, with another seven records having a null status. `TERMINE` is the most common recorded status at 31.90%. I leave unclear codes such as `ORMIG`, `ARCHANT` and `PROCEDURE AMS` uninterpreted for now.

## Date parsing

The loader converts dates it cannot parse into missing values. I read the original date fields as text to separate empty values from parsing failures, then check the earliest and latest dates in each field.

In [16]:
import zipfile

from src.data import ARCHIVE_PATH, CSV_MEMBER, detect_encoding

encoding = detect_encoding(ARCHIVE_PATH)

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    with archive.open(CSV_MEMBER) as csv_file:
        raw_dates = pd.read_csv(
            csv_file,
            sep=";",
            encoding=encoding,
            dtype="string",
            keep_default_na=False,
            usecols=["OBJECTID", "ID_DOSSIER", "DATE_DEPOT", "DATE_MAJ_2"],
        )

In [17]:
date_summary = []

for column in ["DATE_DEPOT", "DATE_MAJ_2"]:
    values = raw_dates[column].str.strip()
    parsed = pd.to_datetime(values, format="%Y%m%d", errors="coerce")

    empty = values.eq("")
    unparsed = ~empty & parsed.isna()

    date_summary.append(
        {
            "field": column,
            "empty": empty.sum(),
            "non_empty_unparsed": unparsed.sum(),
            "parsed": parsed.notna().sum(),
            "earliest": parsed.min(),
            "latest": parsed.max(),
        }
    )

    if unparsed.any():
        print(f"Unparsed values in {column}:")
        display(values[unparsed].value_counts().to_frame(name="records"))

display(pd.DataFrame(date_summary).set_index("field"))

Unparsed values in DATE_DEPOT:


,records
DATE_DEPOT,
20211007105640,1
20220830152321,1
20210130170830,1


,empty,non_empty_unparsed,parsed,earliest,latest
field,,,,,
DATE_DEPOT,68412,3,128809,1995-01-01,2026-09-25
DATE_MAJ_2,36,0,197188,1999-07-13,2026-09-26


`DATE_DEPOT` contains 68,412 empty values and three non-empty values that the loader could not parse. These three values appear to use `YYYYMMDDHHMMSS` rather than the expected `YYYYMMDD` format.

`DATE_MAJ_2` contains 36 empty values and no parsing failures. The successfully parsed filing dates range from 1995-01-01 to 2026-09-25; update dates range from 1999-07-13 to 2026-09-26.

### Filing dates with a time component

I parse the three values using the longer format and inspect the corresponding records. This will show whether the parsing failures explain the missing filing dates among records with descriptions.

In [18]:
values = raw_dates["DATE_DEPOT"].str.strip()
parsed = pd.to_datetime(values, format="%Y%m%d", errors="coerce")

date_issues = raw_dates.loc[
    values.ne("") & parsed.isna(),
    ["OBJECTID", "DATE_DEPOT"],
].rename(columns={"DATE_DEPOT": "raw_date_depot"})

date_issues["parsed_timestamp"] = pd.to_datetime(
    date_issues["raw_date_depot"].str.strip(),
    format="%Y%m%d%H%M%S",
    errors="coerce",
)

date_issues = date_issues.merge(
    df[
        [
            "OBJECTID",
            "ID_DOSSIER",
            "DESCRIPTION",
            "TYPE_OPERATION",
            "ORIGINE",
            "DATE_DEPOT",
        ]
    ],
    on="OBJECTID",
    how="left",
    validate="one_to_one",
)

with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(date_issues)

missing_date_ids = set(with_text.loc[with_text["DATE_DEPOT"].isna(), "OBJECTID"])

print(
    "Exactly the records with text but no parsed filing date:",
    set(date_issues["OBJECTID"]) == missing_date_ids,
)

,OBJECTID,raw_date_depot,parsed_timestamp,ID_DOSSIER,DESCRIPTION,TYPE_OPERATION,ORIGINE,DATE_DEPOT
0,151370,20211007105640,2021-10-07 10:56:40,APAT 316218/1,transformation et rénovation de bureaux au 3ème étage,--,SAD,NaT
1,154303,20220830152321,2022-08-30 15:23:21,APAT 322209/1,installation d'un portillon,--,SAD,NaT
2,147337,20210130170830,2021-01-30 17:08:30,APAT 311661/1,construction d'une véranda,--,SAD,NaT


Exactly the records with text but no parsed filing date: True


The three values parse successfully when the time component is included. They correspond exactly to the three records with descriptions but missing parsed filing dates.

All three are APAT records from SAD, with no operation label (`--`). Two were filed in 2021 and one in 2022. Recovering these dates leaves 68,412 genuinely empty filing dates and no missing filing dates among records with descriptions.

The loader will need to support both date formats. For the calendar splits, I will use the date portion of these timestamps.

In [19]:
date_audit = df[["OBJECTID", "ID_DOSSIER", "DATE_DEPOT", "DATE_MAJ_2", "ORIGINE"]].copy()

recovered_dates = date_issues.set_index("OBJECTID")["parsed_timestamp"].dt.normalize()

date_audit["DATE_DEPOT"] = date_audit["DATE_DEPOT"].fillna(
    date_audit["OBJECTID"].map(recovered_dates)
)

# Download date of the preserved snapshot.
snapshot_date = pd.Timestamp("2026-09-27")

date_checks = pd.DataFrame(
    {
        "filing_after_snapshot": date_audit["DATE_DEPOT"] > snapshot_date,
        "update_after_snapshot": date_audit["DATE_MAJ_2"] > snapshot_date,
        "update_before_filing": (date_audit["DATE_MAJ_2"] < date_audit["DATE_DEPOT"]),
    }
)

display(date_checks.sum().to_frame(name="records"))

for check in date_checks.columns:
    if date_checks[check].any():
        print(check)
        display(date_audit.loc[date_checks[check]].head(20))

,records
filing_after_snapshot,0
update_after_snapshot,0
update_before_filing,0


After including the recovered filing dates, neither date field contains values after the snapshot download date. Among records with both dates available, none has an update date earlier than its filing date.

## Repeated dossier identifiers

Some dossier identifiers appear more than once. I count these repetitions and inspect the most frequent examples to see whether the rows differ in location, description or application details.

A repeated dossier identifier alone is not enough to treat a row as a duplicate.

In [20]:
dossier_counts = df["ID_DOSSIER"].value_counts()
repeated_dossiers = dossier_counts[dossier_counts > 1]

print("Total rows:", len(df))
print("Unique OBJECTID values:", df["OBJECTID"].nunique())
print("Unique ID_DOSSIER values:", df["ID_DOSSIER"].nunique())
print("Dossier identifiers appearing more than once:", len(repeated_dossiers))
print("Rows belonging to those identifiers:", repeated_dossiers.sum())

display(
    repeated_dossiers.value_counts()
    .sort_index()
    .rename_axis("rows_per_identifier")
    .to_frame(name="dossier_identifiers")
)

Total rows: 197224
Unique OBJECTID values: 197224
Unique ID_DOSSIER values: 196879
Dossier identifiers appearing more than once: 343
Rows belonging to those identifiers: 688


,dossier_identifiers
rows_per_identifier,
2,341
3,2


In [21]:
example_ids = repeated_dossiers.head(3).index

columns = [
    "OBJECTID",
    "ID_DOSSIER",
    "TYPE_DOSSIER",
    "NO_DOSSIER",
    "NO_COMPLEMENTAIRE",
    "DATE_DEPOT",
    "DESCRIPTION",
    "TYPE_OPERATION",
    "E",
    "N",
]

examples = df.loc[df["ID_DOSSIER"].isin(example_ids), columns].sort_values(
    ["ID_DOSSIER", "OBJECTID"]
)

with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(examples)

,OBJECTID,ID_DOSSIER,TYPE_DOSSIER,NO_DOSSIER,NO_COMPLEMENTAIRE,DATE_DEPOT,DESCRIPTION,TYPE_OPERATION,E,N
173957,176069,DD 104289/1,DD,104289,1,2011-04-19,construction de 8 abris-bus,EQU,2502339.89,1115585.62
174137,176070,DD 104289/1,DD,104289,1,2011-04-19,construction de 8 abris-bus,EQU,2502547.64,1116495.05
174138,176071,DD 104289/1,DD,104289,1,2011-04-19,construction de 8 abris-bus,EQU,2504056.67,1117466.18
43275,42308,DD 1568,DD,1568,1,NaT,non renseigné,<NA>,2498902.94,1117379.56
43276,42309,DD 1568,DD,1568,1,NaT,non renseigné,<NA>,2498924.8,1117406.63
43277,42310,DD 1568,DD,1568,1,NaT,non renseigné,<NA>,2498946.33,1117436.58
185999,176162,DD 93616/1,DD,93616,1,1996-01-01,"""HOTEL DU RHONE"" transformation des 1er,3ème,5ème et 6ème étage dubâtiment C",--,2499852.68,1117955.9
424,293,DD 93616/1,DD,93616,1,NaT,non renseigné,--,2499851,1117972.66


All 197,224 rows have a unique `OBJECTID`, but there are 196,879 distinct dossier identifiers. Of these, 341 appear twice and two appear three times, covering 688 rows.

The examples show different kinds of repetition. `DD 104289/1` has the same description, filing date and operation code at three different locations. `DD 1568` also appears at three locations, with no description or filing date. For `DD 93616/1`, one row contains a description and filing date while the other has neither.

I keep these rows for now. The examples suggest that repeated identifiers can represent multiple locations or differences in record completeness.

### Differences within repeated dossiers

I check which fields vary within each repeated dossier identifier, counting a missing value and a populated value as different.

I also count rows that match on every field except `OBJECTID`, then repeat the comparison without coordinates. This distinguishes identical record content from records that differ only by location.

In [22]:
repeated_rows = df.loc[df["ID_DOSSIER"].isin(repeated_dossiers.index)]

comparison_columns = [column for column in df.columns if column not in ["OBJECTID", "ID_DOSSIER"]]

distinct_values = repeated_rows.groupby("ID_DOSSIER")[comparison_columns].nunique(dropna=False)

display(
    distinct_values.gt(1)
    .sum()
    .sort_values(ascending=False)
    .to_frame(name="dossiers_with_differences")
)

,dossiers_with_differences
N,343
E,343
DATE_MAJ_2,337
DATE_DEPOT,336
STATUT,336
DESCRIPTION,336
ORIGINE,336
LIEN_SAD,335
TYPE_OPERATION,3
OPERATION,3


In [23]:
content_columns = [column for column in df.columns if column != "OBJECTID"]

non_location_columns = [column for column in content_columns if column not in ["E", "N"]]

print(
    "Extra identical rows, ignoring OBJECTID:",
    df.duplicated(subset=content_columns).sum(),
)

print(
    "Extra matching rows, ignoring OBJECTID and coordinates:",
    df.duplicated(subset=non_location_columns).sum(),
)

Extra identical rows, ignoring OBJECTID: 0
Extra matching rows, ignoring OBJECTID and coordinates: 8


All 343 repeated dossier identifiers have different coordinates. Most also differ in update date, filing date, status, description and origin. These counts overlap: one dossier can differ in several fields.

No rows match on every field except `OBJECTID`. Ignoring coordinates as well produces eight additional matching rows.

Three repeated identifiers have differences in `TYPE_OPERATION` and `OPERATION`, and one has different `NO_DOSSIER` values. I need to inspect these cases before choosing a dossier grouping rule.

### Operation codes and dossier numbers

I inspect repeated identifiers with differences in operation codes, operation descriptions or dossier numbers. I distinguish missing information from conflicting populated values and check whether the identifier agrees with the separate dossier fields.

In [24]:
fields_to_check = ["TYPE_OPERATION", "OPERATION", "NO_DOSSIER"]

needs_review = distinct_values[fields_to_check].gt(1).any(axis=1)
review_ids = distinct_values.index[needs_review]

columns = [
    "OBJECTID",
    "ID_DOSSIER",
    "TYPE_DOSSIER",
    "NO_DOSSIER",
    "NO_COMPLEMENTAIRE",
    "TYPE_OPERATION",
    "OPERATION",
    "DESCRIPTION",
    "ORIGINE",
    "DATE_DEPOT",
    "DATE_MAJ_2",
    "STATUT",
]

review_rows = df.loc[df["ID_DOSSIER"].isin(review_ids), columns].sort_values(
    ["ID_DOSSIER", "OBJECTID"]
)

print("Dossier identifiers to inspect:", len(review_ids))

with pd.option_context(
    "display.max_colwidth",
    None,
    "display.max_columns",
    None,
    "display.max_rows",
    None,
):
    display(review_rows)

Dossier identifiers to inspect: 4


,OBJECTID,ID_DOSSIER,TYPE_DOSSIER,NO_DOSSIER,NO_COMPLEMENTAIRE,TYPE_OPERATION,OPERATION,DESCRIPTION,ORIGINE,DATE_DEPOT,DATE_MAJ_2,STATUT
165379,165163,APA 18919/1,APA,18919,1,--,Non renseigné (valeur par défaut à l'importation des données),installation d'un poste de régulation de gaz,SAD,2001-07-20,2023-11-28,ARCHIVE
6454,7137,APA 18919/1,APA,18919,1,EQU,"Création ou rénovation d' infrastructures, aménagement de terrains ou aménagements divers relevant du domaine public (sans construction de bâtiment",non renseigné,ANC,NaT,2002-03-15,-
135320,126888,DD 105312/2,DD,105312,2,EQU,"Création ou rénovation d'infrastructures, aménagement de terrains ou aménagements divers relevant du domaine public (sans construction de bâtiment)",<NA>,SIT,NaT,NaT,<NA>
163175,165798,DD 105312/2,DD,105312,2,--,Non renseigné (valeur par défaut à l'importation des données),(modération de trafic)modification du projet initial,SAD,2014-11-19,2023-11-28,ACCEPTE
1665,1710,DD 16944/1,DD,16994,1,--,Non renseigné (valeur par défaut à l'importation des données),non renseigné,ANC,NaT,2001-11-29,-
25924,26211,DD 16944/1,DD,16944,1,--,Non renseigné (valeur par défaut à l'importation des données),non renseigné,ANC,NaT,2001-12-06,-
164794,164933,DD 97325/1,DD,97325,1,--,Non renseigné (valeur par défaut à l'importation des données),villa - véranda,SAD,2001-06-25,2023-11-28,ARCHIVE
2430,2431,DD 97325/1,DD,97325,1,VIL,Construction d'une nouvelle villa ou extension importante de villa (création d'un logement supplémentaire),non renseigné,ANC,NaT,2002-03-15,-


The three operation-code differences involve missing information rather than conflicting labels. For `APA 18919/1`, `DD 105312/2` and `DD 97325/1`, one row has a description but no operation label, while another has an operation label but no description.

These are potential candidates for recovering labels across rows of the same dossier. I leave them unchanged until the dossier grouping rule is established.

For `DD 16944/1`, OBJECTID `1710` has `NO_DOSSIER = 16994`, which disagrees with the number in its identifier. The other row has `16944`. I flag this inconsistency without assuming which value is correct.

### Identifier consistency

I compare the type, dossier number and supplementary number in `ID_DOSSIER` with their separate columns.

Some identifiers have no supplementary suffix, such as `DD 1568`. I count these separately and only compare supplementary numbers when the identifier includes one.

In [25]:
identifier_parts = (
    df["ID_DOSSIER"]
    .str.strip()
    .str.extract(
        r"^(?P<dossier_type>\S+)\s+(?P<dossier_number>\d+)"
        r"(?:/(?P<supplementary_number>\d+))?$"
    )
)

identifier_checks = pd.DataFrame(
    {
        "unrecognized_format": identifier_parts["dossier_number"].isna(),
    }
)

field_pairs = {
    "dossier_type": "TYPE_DOSSIER",
    "dossier_number": "NO_DOSSIER",
    "supplementary_number": "NO_COMPLEMENTAIRE",
}

for part, column in field_pairs.items():
    extracted = identifier_parts[part]
    recorded = df[column].str.strip()

    identifier_checks[f"{part}_mismatch"] = (
        extracted.notna() & recorded.notna() & extracted.ne(recorded)
    ).fillna(False)

display(identifier_checks.sum().to_frame(name="records"))

print(
    "Recognized identifiers without a supplementary suffix:",
    (
        identifier_parts["dossier_number"].notna() & identifier_parts["supplementary_number"].isna()
    ).sum(),
)

,records
unrecognized_format,0.0
dossier_type_mismatch,0.0
dossier_number_mismatch,1.0
supplementary_number_mismatch,0.0


Recognized identifiers without a supplementary suffix: 30


In [26]:
flagged_identifiers = df.loc[
    identifier_checks.any(axis=1),
    [
        "OBJECTID",
        "ID_DOSSIER",
        "TYPE_DOSSIER",
        "NO_DOSSIER",
        "NO_COMPLEMENTAIRE",
        "ORIGINE",
    ],
]

print("Flagged rows:", len(flagged_identifiers))
display(flagged_identifiers.head(30))

Flagged rows: 1


,OBJECTID,ID_DOSSIER,TYPE_DOSSIER,NO_DOSSIER,NO_COMPLEMENTAIRE,ORIGINE
1665,1710,DD 16944/1,DD,16994,1,ANC


All dossier identifiers match the expected format. Where both values are present, the dossier type and supplementary number agree with their separate columns.

The only dossier-number mismatch is the previously identified ANC record with OBJECTID `1710`: its identifier contains `16944`, while `NO_DOSSIER` contains `16994`.

### Supplementary applications

I group records by dossier type and dossier number to see how often a base dossier has several supplementary numbers. I include the type because the same number may occur under different dossier types.

I leave the inconsistent record out of this summary and inspect examples before deciding whether this should be the grouping rule for the splits.

In [27]:
consistent_rows = df.loc[~identifier_checks.any(axis=1)]

base_dossiers = consistent_rows.groupby(["TYPE_DOSSIER", "NO_DOSSIER"]).agg(
    records=("OBJECTID", "size"),
    identifiers=("ID_DOSSIER", "nunique"),
    supplementary_numbers=("NO_COMPLEMENTAIRE", "nunique"),
)

multiple_supplements = base_dossiers[base_dossiers["supplementary_numbers"] > 1].sort_values(
    "supplementary_numbers", ascending=False
)

print("Base dossiers in this summary:", len(base_dossiers))
print("Base dossiers with several supplementary numbers:", len(multiple_supplements))
print("Rows in those dossiers:", multiple_supplements["records"].sum())

display(multiple_supplements.head(10))

print(
    "Recognized identifiers without a supplementary suffix:",
    (
        identifier_parts["dossier_number"].notna() & identifier_parts["supplementary_number"].isna()
    ).sum(),
)

Base dossiers in this summary: 183987
Base dossiers with several supplementary numbers: 9456
Rows in those dossiers: 22442


records  identifiers  supplementary_numbers
TYPE_DOSSIER NO_DOSSIER                                             
DD           92130            12           12                     12
             93968            12           12                     12
             107485            9            9                      9
             96148             8            8                      8
             100601            8            8                      8
APA          23014             8            8                      8
DD           106451            8            8                      8
             110335            7            7                      7
             109598            7            7                      7
             93949             7            7                      7

Recognized identifiers without a supplementary suffix: 30


In [28]:
if not multiple_supplements.empty:
    dossier_type, dossier_number = multiple_supplements.index[0]

    example = consistent_rows.loc[
        consistent_rows["TYPE_DOSSIER"].eq(dossier_type)
        & consistent_rows["NO_DOSSIER"].eq(dossier_number),
        [
            "OBJECTID",
            "ID_DOSSIER",
            "NO_COMPLEMENTAIRE",
            "DATE_DEPOT",
            "DESCRIPTION",
            "TYPE_OPERATION",
            "ORIGINE",
        ],
    ]

    with pd.option_context(
        "display.max_colwidth",
        None,
        "display.max_columns",
        None,
        "display.max_rows",
        None,
    ):
        display(example.sort_values(["DATE_DEPOT", "ID_DOSSIER"]))

,OBJECTID,ID_DOSSIER,NO_COMPLEMENTAIRE,DATE_DEPOT,DESCRIPTION,TYPE_OPERATION,ORIGINE
71309,70509,DD 92130/3,3,1996-04-18,(construction saisonnière d'un cinéma en plein air) augmentation enceintesorties de secours et déplacement du local WC,--,SAD
97224,96235,DD 92130/4,4,1997-03-27,(construction saisonnière d'un cinéma en plein air) - agrandissement local WC- modification glacier- déplacement de 2 portes,--,SAD
97943,98230,DD 92130/6,6,1999-03-18,construction saisonnière d'un cinéma en plein air,--,SAD
78256,77432,DD 92130/9,9,2002-03-08,(construction saisonnière d'un cinéma en plein air) -réinstallation sans modification du cinéma pour la saison,--,SAD
95159,94822,DD 92130/10,10,2003-03-20,(construction saisonnière d'un cinéma en plein air) - modification de la structure des gradins et de l'enceinte côté Ouest - agrandissement de la terrasse supérieure - adjonction de la caisse,--,SAD
103122,103819,DD 92130/11,11,2004-03-03,(construction saisonnière d'un cinéma en plein air) - réinstallation du cinéma pour la saison,--,SAD
101656,102564,DD 92130/12,12,2005-03-18,(construction saisonnière d'un cinéma en plein air) -modification de l'enceinte côté est - adjonction d'un élévateur pour fauteuil roulant,--,SAD
75264,74703,DD 92130/13,13,2006-03-09,(construction saisonnière d'un cinéma en plein air) - agrandissementde la terrasse (VIP) côté ouest,--,SAD
105067,105694,DD 92130/14,14,2007-04-05,(construction saisonnière d'un cinéma en plein air) - modification de la structure bois par du métal,--,SAD
86705,85927,DD 92130/15,15,2008-03-12,(construction saisonnière d'un cinéma en plein air) - reconstruction identique à la saison 2007 -,--,SAD


Excluding the inconsistent record, there are 183,987 base dossiers identified by dossier type and number. Of these, 9,456 have multiple supplementary numbers, covering 22,442 rows.

`DD 92130` illustrates this structure: its 12 entries concern a seasonal open-air cinema, with dated applications from 1996 to 2009 describing modifications or annual reinstallation.

For the split audit, I will use `(TYPE_DOSSIER, NO_DOSSIER)` as a provisional grouping key and retain the individual applications as separate records. I still need to check how these groups cross the proposed time boundaries.

Thirty identifiers have no supplementary suffix. I inspect these alongside the record with a missing `NO_COMPLEMENTAIRE` before finishing the identifier audit.

### Missing supplementary numbers

I check whether identifiers without a suffix have a supplementary number in the separate column. I also inspect the record where that column is missing, to see whether the identifier supplies the value.

In [29]:
no_suffix = identifier_parts["supplementary_number"].isna()
no_supplement = df["NO_COMPLEMENTAIRE"].isna()

print("Supplementary values for identifiers without a suffix:")
display(df.loc[no_suffix, "NO_COMPLEMENTAIRE"].value_counts(dropna=False).to_frame(name="records"))

columns = [
    "OBJECTID",
    "ID_DOSSIER",
    "TYPE_DOSSIER",
    "NO_DOSSIER",
    "NO_COMPLEMENTAIRE",
    "ORIGINE",
    "DATE_DEPOT",
]

supplement_review = df.loc[no_suffix | no_supplement, columns].copy()
supplement_review["has_description"] = ~no_text.loc[supplement_review.index]

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
):
    display(supplement_review.sort_values("ID_DOSSIER"))

Supplementary values for identifiers without a suffix:


,records
NO_COMPLEMENTAIRE,
1,29
<NA>,1


,OBJECTID,ID_DOSSIER,TYPE_DOSSIER,NO_DOSSIER,NO_COMPLEMENTAIRE,ORIGINE,DATE_DEPOT,has_description
43282,42315,APA 10549,APA,10549,1,ANC,NaT,False
64307,65254,APA 1199,APA,1199,1,ANC,NaT,False
43285,42318,APA 146,APA,146,1,ANC,NaT,False
43286,42319,APA 1754,APA,1754,1,ANC,NaT,False
63936,66225,APA 6343,APA,6343,1,ANC,NaT,False
66993,68409,APA 653,APA,653,1,ANC,NaT,False
43284,42317,APA 7098,APA,7098,1,ANC,NaT,False
66983,68399,APA 9937,APA,9937,1,ANC,NaT,False
43206,42305,DD 1338,DD,1338,1,ANC,NaT,False
43275,42308,DD 1568,DD,1568,1,ANC,NaT,False


Thirty rows have an identifier without a supplementary suffix. In 29 cases, `NO_COMPLEMENTAIRE` contains `1`. For `DD 55067` (OBJECTID `68408`), the supplementary number is missing from both fields, so I leave it unknown.

All 30 rows come from ANC and have no description or filing date. They therefore fall outside the description-based modelling population. Their dossier type and number remain available for grouping.

## Operation categories

I count each operation code alongside its recorded definition, keeping missing values and wording variants visible. I also count how many rows in each combination have a description.

This checks how consistently the export records the categories before I define the eligible labelled population.

In [30]:
operation_definitions = (
    df.assign(has_description=~no_text)
    .groupby(["TYPE_OPERATION", "OPERATION"], dropna=False)
    .agg(
        records=("OBJECTID", "size"),
        with_description=("has_description", "sum"),
    )
    .reset_index()
    .sort_values(
        ["TYPE_OPERATION", "records"],
        ascending=[True, False],
    )
)

with pd.option_context(
    "display.max_colwidth",
    None,
    "display.max_rows",
    None,
):
    display(operation_definitions)

,TYPE_OPERATION,OPERATION,records,with_description
0,--,Non renseigné (valeur par défaut à l'importation des données),147397,79022
1,AFF,"Changement d'affectation important, avec ou sans rénovation",555,555
2,AGR,"Création de surfaces supplémentaires destinées à l'habitat ou aux activités (combles, surélévation, extension, ...)",2325,2324
3,AGV,"Création de surfaces supplémentaires destinées à l'habitat (combles, surélévation, extension) dans une villa",1992,1992
4,AM,"Opération d'aménagement (modification de zone, morcellement...) et de rénovation de périmètres urbains",38,38
6,EQU,"Création ou rénovation d'infrastructures, aménagement de terrains ou aménagements divers relevant du domaine public (sans construction de bâtiment)",4550,4549
5,EQU,"Création ou rénovation d' infrastructures, aménagement de terrains ou aménagements divers relevant du domaine public (sans construction de bâtiment",1,0
7,NOU,Construction nouvelle importante ou extension importante (création d'un bâtiment supplémentaire),4015,4011
8,PI,"Construction neuve de peu d'importance, isolée ou attenante au bâtiment principal, relevant du domaine privé",14815,14815
9,REN,"Travaux d'entretien et de rénovation intérieure ou extérieure, sans création de surfaces supplémentaires destinées à l'habitat ou aux activités et sans changement d'affectation important",17361,17361


Each recorded operation code has one definition, except `EQU`. Its second wording contains an extra space and lacks the closing parenthesis; it occurs once, on a row without a description.

The placeholder `--` consistently accompanies the missing-operation definition. Of the 35 rows with a null operation code, 32 have that same placeholder definition and three have no definition. None has a description.

Eight rows have a target-class label but no description: one AGR, two EQU, four NOU and one VIL. These cannot be used directly for description-based training.

AM is a recorded category covering planning operations such as zoning changes, subdivision and redevelopment of urban areas. It is outside the configured eight-class target and remains separate from the unlabelled records. Its exclusion does not establish that the unlabelled pool contains only the eight target categories.

## Label eligibility

I count labelled records using explicit membership in the eight target classes. I separate records without descriptions, AM records, unlabelled records and any unexpected codes.

These counts describe the population before deduplication or splitting. Records with a description and a target label are candidates for supervised learning, subject to the remaining audit.

In [31]:
target_classes = ["AFF", "AGR", "AGV", "EQU", "NOU", "PI", "REN", "VIL"]

has_text = ~no_text
target_label = df["TYPE_OPERATION"].isin(target_classes)
am_label = df["TYPE_OPERATION"].isin(["AM"])
missing_label = df["TYPE_OPERATION"].isna() | df["TYPE_OPERATION"].isin(["--"])
other_label = ~(target_label | am_label | missing_label)

population_counts = pd.Series(
    {
        "Without description": no_text.sum(),
        "Description and target label": (has_text & target_label).sum(),
        "Description and AM label": (has_text & am_label).sum(),
        "Description but no label": (has_text & missing_label).sum(),
        "Description and unexpected code": (has_text & other_label).sum(),
    },
    name="records",
)

display(population_counts.to_frame())

assert population_counts.sum() == len(df), "Population counts do not reconcile."

,records
Without description,68418
Description and target label,49746
Description and AM label,38
Description but no label,79022
Description and unexpected code,0


In [32]:
display(
    df.loc[has_text & target_label, "TYPE_OPERATION"]
    .value_counts()
    .reindex(target_classes, fill_value=0)
    .to_frame(name="records_with_description")
)

,records_with_description
TYPE_OPERATION,
AFF,555
AGR,2324
AGV,1992
EQU,4549
NOU,4011
PI,14815
REN,17361
VIL,4139


There are 49,746 records with a description and one of the eight target labels, 79,022 with a description but no label, and 38 with a description and an AM label. The remaining 68,418 records have no description. These groups account for every row, with no unexpected operation codes among records with descriptions.

The labelled classes are unevenly represented: REN has 17,361 records and PI has 14,815, while AFF has 555. These are row counts before deduplication, grouping or splitting.

The 79,022 unlabelled records remain candidates for review. A description alone does not establish that a record fits the target taxonomy.

## Description length

I measure description length in characters and whitespace-separated words, then compare character lengths across the target classes, AM and unlabelled records.

Word counts give a rough indication of length. Model sequence lengths will need a separate check with the model's tokenizer.

In [33]:
text_audit = df.loc[
    has_text,
    ["OBJECTID", "ID_DOSSIER", "DESCRIPTION", "TYPE_OPERATION"],
].copy()

text_audit["label_group"] = (
    text_audit["TYPE_OPERATION"].fillna("Unlabelled").replace({"--": "Unlabelled"})
)

text_audit["characters"] = text_audit["DESCRIPTION"].str.len()
text_audit["words"] = text_audit["DESCRIPTION"].str.split().str.len()

display(text_audit[["characters", "words"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))

,characters,words
count,128806.0,128806.0
mean,64.5,8.9
std,44.2,6.3
min,3.0,1.0
50%,51.0,7.0
90%,123.0,17.0
95%,160.0,22.0
99%,236.0,33.0
max,254.0,46.0


In [34]:
display(
    text_audit.groupby("label_group")["characters"]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(1)
)

print(
    "Descriptions with exactly 254 characters:",
    text_audit["characters"].eq(254).sum(),
)

print(
    "Descriptions longer than 254 characters:",
    text_audit["characters"].gt(254).sum(),
)

,count,mean,std,min,50%,90%,95%,99%,max
label_group,,,,,,,,,
AFF,555.0,69.0,30.7,19.0,65.0,100.2,123.6,191.1,254.0
AGR,2324.0,73.4,42.9,7.0,62.0,130.0,157.8,227.0,254.0
AGV,1992.0,64.4,40.3,5.0,54.0,118.0,142.4,209.4,254.0
AM,38.0,47.3,22.9,17.0,43.0,82.5,90.6,103.2,105.0
EQU,4549.0,55.9,34.0,7.0,46.0,98.0,121.0,184.5,254.0
NOU,4011.0,90.1,53.4,5.0,77.0,166.0,201.0,254.0,254.0
PI,14815.0,40.6,26.5,3.0,37.0,73.0,89.0,131.0,254.0
REN,17361.0,57.3,30.6,6.0,49.0,96.0,117.0,170.0,254.0
Unlabelled,79022.0,68.2,46.8,3.0,53.0,133.0,172.0,253.8,254.0


Descriptions with exactly 254 characters: 917
Descriptions longer than 254 characters: 0
